# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
# imports
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI
import gradio as gr # oh yeah!

In [ ]:
# constants

MODEL_GPT = 'openai/gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [ ]:
# set up environment
#connecting to openrouter
load_dotenv(override=True)
api_key = os.getenv('OPENROUTER_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-or-v1"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

In [ ]:
ODEL_GPT = "openai/gpt-4o-mini"
MODEL_LLAMA = "llama3.2"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

AI CODE Explainer


In [ ]:
def chat_with_model(user_message, model_choice, language):
    
    system_prompt = f"""
You are a senior software engineer.
Explain technical concepts clearly with examples.

IMPORTANT:
Respond ONLY in {language}.
Use simple and clear language suitable for a learner.
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    # If using GPT (OpenRouter)
    if model_choice == "GPT (OpenRouter)":
        response = client.chat.completions.create(
            model=MODEL_GPT,
            messages=messages,
            temperature=0.3,
        )
        return response.choices[0].message.content

    # If using LLaMA (Ollama)
    elif model_choice == "LLaMA (Ollama)":
        import ollama
        response = ollama.chat(
            model=MODEL_LLAMA,
            messages=messages,
        )
        return response["message"]["content"]

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## AI Code Explainer")

    user_input = gr.Textbox(
        label="Ask your question",
        lines=6,
        placeholder="Paste your Python code or ask something..."
    )

    model_dropdown = gr.Dropdown(
        choices=["GPT (OpenRouter)", "LLaMA (Ollama)"],
        value="GPT (OpenRouter)",
        label="Choose Model"
    )

    language_dropdown = gr.Dropdown(
        choices=["English", "French", "Spanish"],
        value="English",
        label="Response Language"
    )

    output = gr.Markdown(label="Response")

    submit_btn = gr.Button("Explain")

    submit_btn.click(
        fn=chat_with_model,
        inputs=[user_input, model_dropdown, language_dropdown],
        outputs=output,
    )

demo.launch()

Research Paper Explainer


In [ ]:
from pypdf import PdfReader

def extract_pdf_text(file):
    reader = PdfReader(file.name)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

In [ ]:
def chunk_text(text, chunk_size=4000):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

In [ ]:
def analyze_paper(file, model_choice):

    print("=== analyze_paper called ===")

    if file is None:
        print("No file uploaded.")
        return "Please upload a PDF file."

    print(f"File received: {file.name}")

    # Step 1: Extract text
    try:
        full_text = extract_pdf_text(file)
        print(f"Extracted text length: {len(full_text)} characters")
    except Exception as e:
        print("Error during PDF extraction:", e)
        return "Error extracting PDF text."

    if not full_text.strip():
        print("Warning: Extracted text is empty.")
        return "No readable text found in PDF."

    # Step 2: Chunk text
    chunks = chunk_text(full_text)
    print(f"Total chunks created: {len(chunks)}")

    if len(chunks) == 0:
        return "Text chunking failed."

    system_prompt = """
You are a research scientist.
Explain the technical content of this research paper clearly.

Structure your response as:
1. Problem
2. Method
3. Technical details
4. Experiments
5. Strengths
6. Limitations
"""

    combined_summary = ""

    # Limit chunks for debugging
    for i, chunk in enumerate(chunks[:3]):
        print(f"\n--- Sending chunk {i+1} to model ---")
        print(f"Chunk length: {len(chunk)}")

        try:
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": chunk},
            ]

            response = client.chat.completions.create(
                model=MODEL_GPT,
                messages=messages,
                temperature=0.2,
            )

            content = response.choices[0].message.content
            print(f"Received response for chunk {i+1}")
            combined_summary += content + "\n\n"

        except Exception as e:
            print(f"Error during model call for chunk {i+1}:", e)
            return "Model call failed."

    print("=== analyze_paper finished ===")
    return combined_summary

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## Research Paper Explainer")

    file_input = gr.File(
        label="Upload Research Paper (PDF only)",
        file_types=[".pdf"]
    )

    model_dropdown = gr.Dropdown(
        choices=["GPT (OpenRouter)","LLaMA (Ollama)"],
        value="GPT (OpenRouter)",
        label="Choose Model"
    )

    output = gr.Markdown()

    analyze_btn = gr.Button("Analyze Paper")

    analyze_btn.click(
        fn=analyze_paper,
        inputs=[file_input, model_dropdown],
        outputs=output,
    )

demo.launch()